# Model failure browser (dev)

Every dev example with the full passage, gold answers, slice tags and every model's prediction per seed.
Source: `report/assets/tables/per_example_all_models.jsonl`, built by `python experiments/global/run.py mrc.analysis.per_example`
(rerun it after adding a model to `experiments/global/configs/models_registry.yaml`).

Thresholds are tuned per run on dev (same as `report/assets/tables/main_table.md`), so scores are dev-optimistic.
Partial API runs (`llm_*_api`) cover only the first 22 to 25 dev ids and are untuned.
`itest` is not loaded.

Scope: this notebook compares models question by question. Per-experiment analysis (outcome tables, slices, seed stability, calibration, error samples for manual labels) is in `experiments/NN-*/analysis/` and each `summary.md`; aggregated cross-model tables are in `report/assets/tables/`. See `eda/README.md`.

In [1]:
import json, re, html, collections
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import HTML, display

pd.set_option("display.max_colwidth", 80)

def find_root(p=Path.cwd()):
    for d in [p, *p.parents]:
        if (d / "report/assets/tables/per_example_all_models.jsonl").exists():
            return d
    raise FileNotFoundError("run: python experiments/global/run.py mrc.analysis.per_example")

ROOT = find_root()
R = [json.loads(l) for l in open(ROOT / "report/assets/tables/per_example_all_models.jsonl", encoding="utf-8")]
BY_ID = {r["id"]: r for r in R}
MODELS = list(R[0]["models"])
TRIVIAL = ["no_answer", "sliding_window", "sliding_window_thr"]
MAIN = [m for m in MODELS if m not in TRIVIAL]
print(len(R), "dev examples; models:", MODELS)

803 dev examples; models: ['no_answer', 'sliding_window', 'sliding_window_thr', 'classical_pipeline', 'bidaf_noattn', 'bidaf', 'plm_indobert_full']


In [2]:
# Flat table: one row per example, per-model mean F1 (over seeds) and seed13 category / pred
rows = []
for r in R:
    row = {"id": r["id"], "question": r["question"], "is_impossible": r["is_impossible"],
           "gold": " | ".join(a["text"] for a in r["answers"]) or "<no answer>", **r["slices"]}
    row["ends_space_q"] = r["question"].endswith(" ?")
    for m, v in r["models"].items():
        s = v["seeds"].get("seed13") or next(iter(v["seeds"].values()))
        row[f"{m}.f1"] = v["mean_f1"]
        row[f"{m}.cat"] = s["category"]
        row[f"{m}.pred"] = s["pred"] or "<no answer>"
    for m, v in r["partial_models"].items():
        row[f"{m}.f1"] = v["f1"]; row[f"{m}.cat"] = v["category"]; row[f"{m}.pred"] = v["pred"] or "<no answer>"
    rows.append(row)
df = pd.DataFrame(rows).set_index("id")
df[["question", "gold"] + [f"{m}.f1" for m in MAIN]].head()

,question,gold,classical_pipeline.f1,bidaf_noattn.f1,bidaf.f1,plm_indobert_full.f1
id,,,,,,
qa_006703b670a2de86a02f,kapankah Uni Eropa didirikan?,1993,1.0,0.0741,0.0000,1.0000
qa_00e198cdbe7a6db9fff7,Kapan Yan Xishan diangkat menjadi panglima perang Tiongkok ?,1911,0.0,0.0000,0.0000,0.6667
qa_00ff25c8418d611dec83,Kapan Barthelemy Boganda mengundurkan diri dari politikus?,<no answer>,1.0,1.0000,1.0000,1.0000
qa_01364a9092f96eb6e2be,Apakah kota terbesar di Iran?,Teheran,1.0,0.0000,0.3333,0.6667
qa_019cc3e2fe995662f0ec,Apa yang dimaksud dengan ELS?,<no answer>,1.0,0.6667,1.0000,1.0000


## Overview
Mean F1 per model, error category counts (seed 13), and pairwise agreement.

In [3]:
display(df[[f"{m}.f1" for m in MODELS]].mean().mul(100).round(2).rename("F1 (mean over seeds)").to_frame())
cats = pd.DataFrame({m: df[f"{m}.cat"].value_counts() for m in MAIN}).fillna(0).astype(int)
cats

,F1 (mean over seeds)
no_answer.f1,46.20
sliding_window.f1,46.20
sliding_window_thr.f1,46.81
classical_pipeline.f1,53.36
bidaf_noattn.f1,56.47
bidaf.f1,56.61
plm_indobert_full.f1,73.53


,classical_pipeline,bidaf_noattn,bidaf,plm_indobert_full
boundary_too_long,6,30,47,27
boundary_too_short,30,12,13,21
correct,412,428,421,560
false_answer_on_unanswerable,63,39,22,35
false_no_answer,204,229,248,106
partial_overlap,10,16,11,15
wrong_sentence,43,28,29,24
wrong_span_same_sentence,35,21,12,15


In [4]:
# Pairwise: how often A is right (mean F1 >= 0.5) where B is wrong, and vice versa
ok = {m: df[f"{m}.f1"] >= 0.5 for m in MAIN}
pair = pd.DataFrame({a: {b: int((ok[a] & ~ok[b]).sum()) for b in MAIN} for a in MAIN}).T
print("cell [A, B] = #examples A gets right and B gets wrong")
pair

cell [A, B] = #examples A gets right and B gets wrong


,classical_pipeline,bidaf_noattn,bidaf,plm_indobert_full
classical_pipeline,0,113,98,42
bidaf_noattn,132,0,40,38
bidaf,129,52,0,40
plm_indobert_full,214,191,181,0


## Rendering helpers

- **green**: gold answer span(s) in the context
- **yellow**: context word that also appears in the question
- **orange** (in the question): question word that does not occur in the context
- Under the passage, each model's prediction per seed, shown in a window of the passage with the predicted span **boxed**.

In [5]:
GREEN, YELLOW, ORANGE = "#8be28b", "#fff176", "#ffb74d"
WORD = re.compile(r"\w+", re.UNICODE)
CAT_COLOR = {"correct": "#2e7d32", "false_no_answer": "#c62828", "false_answer_on_unanswerable": "#ad1457",
             "boundary_too_long": "#ef6c00", "boundary_too_short": "#f9a825", "partial_overlap": "#8d6e63",
             "wrong_span_same_sentence": "#6a1b9a", "wrong_sentence": "#283593"}

def _runs(text, color):
    out, i = [], 0
    while i < len(text):
        j = i
        while j < len(text) and color[j] == color[i]:
            j += 1
        seg = html.escape(text[i:j])
        out.append(f'<mark style="background:{color[i]};color:#000">{seg}</mark>' if color[i] else seg)
        i = j
    return "".join(out)

def highlight_context(r):
    ctx, q = r["context"], r["question"]
    q_words = {w.lower() for w in WORD.findall(q)}
    color = [None] * len(ctx)
    for m in WORD.finditer(ctx):
        if m.group().lower() in q_words:
            color[m.start():m.end()] = [YELLOW] * (m.end() - m.start())
    for a in r["answers"]:
        st = a["answer_start"]
        for k in range(st, min(st + len(a["text"]), len(ctx))):
            color[k] = GREEN
    return _runs(ctx, color)

def highlight_question(r):
    c_words = {w.lower() for w in WORD.findall(r["context"])}
    q, out, last = r["question"], [], 0
    for m in WORD.finditer(q):
        out.append(html.escape(q[last:m.start()]))
        w = html.escape(m.group())
        out.append(f'<mark style="background:{ORANGE};color:#000">{w}</mark>' if m.group().lower() not in c_words else w)
        last = m.end()
    out.append(html.escape(q[last:]))
    return "".join(out)

def span_window(ctx, pred, start, w=70):
    if not pred:
        return "<i>no answer</i>"
    if start is None or start < 0 or ctx[start:start + len(pred)] != pred:
        start = ctx.find(pred)
    if start < 0:
        return f"<i>not a passage span:</i> {html.escape(pred)}"
    a, b = max(0, start - w), min(len(ctx), start + len(pred) + w)
    return (("..." if a else "") + html.escape(ctx[a:start])
            + f'<span style="outline:2px solid #1565c0;background:#e3f2fd;color:#000">{html.escape(pred)}</span>'
            + html.escape(ctx[start + len(pred):b]) + ("..." if b < len(ctx) else ""))

def _badge(cat):
    return f'<span style="color:#fff;background:{CAT_COLOR.get(cat, "#555")};padding:0 4px;border-radius:3px;font-size:85%">{cat}</span>'

def show(ids, models=None, seeds=("seed13",), partial=True):
    """Render full instances. ids: list of ids or a DataFrame/Index. seeds=None shows all seeds."""
    if isinstance(ids, (pd.DataFrame, pd.Series)):
        ids = ids.index
    models = models or MAIN
    parts = []
    for n, i in enumerate(ids, 1):
        r = BY_ID[i]
        tag = "UNANSWERABLE" if r["is_impossible"] else "answerable"
        gold = " | ".join(a["text"] for a in r["answers"]) or "&lt;no answer&gt;"
        sl = r["slices"]
        meta = ", ".join(f"{k}={sl.get(k)}" for k in ("q_type", "ans_type", "ans_len_bucket", "lex_overlap_bucket", "ctx_len_bucket") if k in sl)
        lines = []
        for m in models:
            for s, v in r["models"][m]["seeds"].items():
                if seeds and s not in seeds:
                    continue
                lines.append(f'<tr><td><b>{m}</b> <small>{s}</small></td><td>{_badge(v["category"])} '
                             f'<small>F1 {v["f1"]:.2f}</small></td><td>{span_window(r["context"], v["pred"], v["start"])}</td></tr>')
        if partial:
            for m, v in r["partial_models"].items():
                raw = html.escape((v.get("raw_output") or "")[:80])
                lines.append(f'<tr><td><b>{m}</b> <small>{v.get("grounding")}</small></td><td>{_badge(v["category"])} '
                             f'<small>F1 {v["f1"]:.2f}</small></td><td>{span_window(r["context"], v["pred"], v["start"])}'
                             f'<br><small>raw: {raw}</small></td></tr>')
        parts.append(
            f'<div style="margin:0 0 16px;padding:8px;border:1px solid #888;border-radius:6px">'
            f'<b>{n}. {highlight_question(r)}</b> <i>({tag})</i> <small><code>{i}</code></small><br>'
            f'<small>gold: {html.escape(gold) if r["answers"] else gold} &nbsp; [{html.escape(meta)}]</small>'
            f'<div style="margin:6px 0">{highlight_context(r)}</div>'
            f'<table style="font-size:90%;text-align:left">{"".join(lines)}</table></div>')
    display(HTML("".join(parts)))

## Side-by-side model answers

One row per example, one column per model (seed 13; partial API LLM runs where available). Cell colour = F1 of that answer
(green 1.0, through yellow, to red 0.0); grey italic = the model abstained. The passage is collapsed under each question.

**Ordering:** examples where *every* model gave an answer (no abstention) come first, because they show competing spans
directly. Priority tiers:
1. all main models **and** both API LLMs answered (only possible on the ~22 ids the API runs cover)
2. all main models answered
3. the rest, by number of models that answered

In [6]:
API = ["llm_nemotron35_api", "llm_gemini35flash_api"]

def answers_of(r, models=MAIN, partial=API):
    out = {}
    for m in models:
        v = r["models"][m]["seeds"].get("seed13") or next(iter(r["models"][m]["seeds"].values()))
        out[m] = v
    for m in partial:
        if m in r["partial_models"]:
            out[m] = r["partial_models"][m]
    return out

def tier(r):
    a = answers_of(r)
    main_ans = all(a[m]["pred"] for m in MAIN)
    api_ans = all(m in a and a[m]["pred"] for m in API)
    n_ans = sum(bool(v["pred"]) for v in a.values())
    return (1 if main_ans and api_ans else 2 if main_ans else 3), -n_ans

PRIORITY = sorted(BY_ID, key=lambda i: tier(BY_ID[i]))
tiers = pd.Series([tier(BY_ID[i])[0] for i in PRIORITY]).value_counts().sort_index()
print("examples per tier:", tiers.to_dict())

def _f1_color(f):
    # red (0) -> yellow (0.5) -> green (1)
    r_, g_ = (230, int(80 + 340 * f)) if f < .5 else (int(230 - 300 * (f - .5)), 200)
    return f"rgb({min(r_,255)},{min(g_,255)},110)"

def side_by_side(ids, models=None, show_context=True):
    """Table: rows = examples, columns = model answers. ids: list, Index or DataFrame."""
    if isinstance(ids, (pd.DataFrame, pd.Series)):
        ids = ids.index
    cols = models or MAIN + API
    head = "".join(f'<th style="min-width:120px">{m}</th>' for m in cols)
    body = []
    for n, i in enumerate(ids, 1):
        r = BY_ID[i]
        a = answers_of(r)
        gold = " | ".join(x["text"] for x in r["answers"]) or "<no answer>"
        q = highlight_question(r)
        if show_context:
            q += f'<details><summary><small>passage</small></summary><div style="font-size:85%">{highlight_context(r)}</div></details>'
        cells_ = []
        for m in cols:
            if m not in a:
                cells_.append('<td style="color:#999">n/a</td>')
            elif not a[m]["pred"]:
                cells_.append(f'<td style="background:{_f1_color(a[m]["f1"])};color:#333"><i>(no answer)</i></td>')
            else:
                cells_.append(f'<td style="background:{_f1_color(a[m]["f1"])};color:#000" title="{a[m]["category"]}">'
                              f'{html.escape(a[m]["pred"])}<br><small>{a[m]["category"]}</small></td>')
        body.append(f'<tr><td>{n}</td><td style="max-width:360px">{q}<br><small><code>{i}</code> tier {tier(r)[0]}</small></td>'
                    f'<td><b>{html.escape(gold)}</b></td>{"".join(cells_)}</tr>')
    display(HTML('<table style="border-collapse:collapse;font-size:90%;text-align:left" border="1">'
                 f'<tr><th>#</th><th>question</th><th>gold</th>{head}</tr>{"".join(body)}</table>'))

examples per tier: {1: 1, 2: 91, 3: 711}


In [7]:
# Tier 1 and 2 first: every main model answered (API LLMs too where they ran)
side_by_side(PRIORITY[:25])

#,question,gold,classical_pipeline,bidaf_noattn,bidaf,plm_indobert_full,llm_nemotron35_api,llm_gemini35flash_api
1,Dimana letak Düsseldorf?passageDüsseldorf adalah ibu kota Negara Bagian Nordrhein-Westfalen di Jerman dan (bersama-sama dengan Köln dan Wilayah Ruhr) merupakan pusat ekonomi Jerman bagian Barat. Düsseldorf terletak di tepi Sungai Rhein dan merupakan salah satu pusat utama dari daerah Rhein-Ruhr yang padat penduduknya.qa_05472ac12e5fdcef4ee0 tier 1,Jerman,Sungai Rheinwrong_sentence,pusat ekonomi Jerman bagian Baratboundary_too_long,Jerman bagian Baratboundary_too_long,tepi Sungai Rheinwrong_sentence,Düsseldorf terletak di tepi Sungai Rheinwrong_sentence,tepi Sungai Rheinwrong_sentence
2,"kapankah Uni Eropa didirikan?passageMasyarakat Ekonomi Eropa (European Economic Community) adalah organisasi kawasan yang bertujuan menyatukan ekonomi negara-negara anggotanya. Organisasi ini dibentuk melalui Perjanjian Roma tahun 1957.[1] Setelah Uni Eropa (UE) dibentuk tahun 1993, MEE disatukan dan berganti nama menjadi Masyarakat Eropa (EC). Pada tahun 2009, semua lembaga ME dileburkan menjadi Uni Eropa.qa_006703b670a2de86a02f tier 2",1993,1993correct,1957wrong_sentence,1957wrong_sentence,1993correct,tahun 1993boundary_too_long,(no answer)
3,"Siapakah nama pemimpin dalam gereja ?passagePaus (dari Dutch: paus; Latin: papa dari Greek: πάππας pappas,[1] ""ayah"")[2] adalah Uskup Roma dan pemimpin Gereja Katolik di seluruh dunia.[3] Keutamaan Uskup Roma sebagian besar berasal dari peranannya dalam tradisi sebagai penerus Santo Petrus, kepada siapa Yesus memberikan kunci Surga dan kuasa untuk ""mengikat dan melepaskan"" serta menyebutnya sebagai ""batu karang"" yang di atasnya Gereja kemudian dibangun. Paus saat ini adalah Paus Fransiskus, terpilih pada tanggal 13 Maret 2013, menggantikan Paus Benediktus XVI.[4]qa_0986961735e0c79ff572 tier 2",Paus,Pauscorrect,Fransiskuswrong_sentence,Paus Fransiskusboundary_too_long,Pauscorrect,Pauscorrect,(no answer)
4,"Berapa luas Papua?passagePapua adalah sebuah provinsi terluas Indonesia yang terletak di bagian tengah Pulau Papua atau bagian paling timur wilayah Papua milik Indonesia. Belahan timurnya merupakan negara Papua Nugini. Provinsi Papua dulu mencakup seluruh wilayah Papua Bagian barat, namun sejak tahun 2003 dibagi menjadi dua provinsi dengan bagian timur tetap memakai nama Papua sedangkan bagian baratnya memakai nama Papua Barat. Papua memiliki luas 808.105km persegi dan merupakan pulau terbesar kedua di dunia dan terbesar pertama di Indonesia.qa_0a3867b0e2c4601ee016 tier 2",808.105km,808.105km persegiboundary_too_long,808.105km persegiboundary_too_long,808.105km persegiboundary_too_long,808.105km persegiboundary_too_long,808.105km persegiboundary_too_long,n/a
5,"Siapa yang menemukan Tabel periodik?passageSejarah tabel periodik mencerminkan perkembangan pemahaman sifat-sifat kimia selama lebih dari satu abad. Peristiwa terpenting dalam sejarahnya terjadi pada tahun 1869, ketika tabel dipublikasikan oleh Dmitri Mendeleev,[1] yang mengembangkan tabel berdasarkan pengembangan terdahulu oleh ilmuwan seperti Antoine-Laurent de Lavoisier dan John Newlands, tetapi hanya Mendeleev yang mendapat kredit untuk pengembangannya.qa_0bf9cafef11db24ae688 tier 2",Dmitri Mendeleev,Dmitri Mendeleevcorrect,Dmitri Mendeleevcorrect,"Dmitri Mendeleev,[1] yang mengembangkan tabel berdasarkan pengembangan terdahulu oleh ilmuwan seperti Antoine-Laurent de Lavoisier dan John Newlandsboundary_too_long",Dmitri Mendeleevcorrect,n/a,n/a
6,"apakah nama ibukota Vietnam?passageIbukota Vietnam adalah Hanoi (dahulu berfungsi sebagai ibukota Vietnam Utara), sedangkan kota terbesar dan terpadat adalah Kota Ho Chi Minh (dahulu dikenal sebagai Saigon). provinsi (dalam Bahasa Vietnam di sebut tỉnh) dan 5 kotamadya yang di kontrol langsung oleh pemerintah pusat dan memiliki level yang sama dengan provinsi (thành phố trực thuộc trung ương). Ke-59 provinsi-provinsi tersebut kemudian dibagi-bagi menjadi kotamadya provinsi (thành 

In [ ]:
# Only the disagreements among examples where all main models answered: at least one right and one wrong
all_answered = [i for i in PRIORITY if tier(BY_ID[i])[0] <= 2]
def f1s(i): return [answers_of(BY_ID[i])[m]["f1"] for m in MAIN]
disagree = [i for i in all_answered if max(f1s(i)) >= .5 and min(f1s(i)) < .5]
print(len(all_answered), "all-answered;", len(disagree), "with disagreement")



92 all-answered; 59 with disagreement


#,question,gold,classical_pipeline,bidaf_noattn,bidaf,plm_indobert_full,llm_nemotron35_api,llm_gemini35flash_api
1,Dimana letak Düsseldorf?passageDüsseldorf adalah ibu kota Negara Bagian Nordrhein-Westfalen di Jerman dan (bersama-sama dengan Köln dan Wilayah Ruhr) merupakan pusat ekonomi Jerman bagian Barat. Düsseldorf terletak di tepi Sungai Rhein dan merupakan salah satu pusat utama dari daerah Rhein-Ruhr yang padat penduduknya.qa_05472ac12e5fdcef4ee0 tier 1,Jerman,Sungai Rheinwrong_sentence,pusat ekonomi Jerman bagian Baratboundary_too_long,Jerman bagian Baratboundary_too_long,tepi Sungai Rheinwrong_sentence,Düsseldorf terletak di tepi Sungai Rheinwrong_sentence,tepi Sungai Rheinwrong_sentence
2,"kapankah Uni Eropa didirikan?passageMasyarakat Ekonomi Eropa (European Economic Community) adalah organisasi kawasan yang bertujuan menyatukan ekonomi negara-negara anggotanya. Organisasi ini dibentuk melalui Perjanjian Roma tahun 1957.[1] Setelah Uni Eropa (UE) dibentuk tahun 1993, MEE disatukan dan berganti nama menjadi Masyarakat Eropa (EC). Pada tahun 2009, semua lembaga ME dileburkan menjadi Uni Eropa.qa_006703b670a2de86a02f tier 2",1993,1993correct,1957wrong_sentence,1957wrong_sentence,1993correct,tahun 1993boundary_too_long,(no answer)
3,"Siapakah nama pemimpin dalam gereja ?passagePaus (dari Dutch: paus; Latin: papa dari Greek: πάππας pappas,[1] ""ayah"")[2] adalah Uskup Roma dan pemimpin Gereja Katolik di seluruh dunia.[3] Keutamaan Uskup Roma sebagian besar berasal dari peranannya dalam tradisi sebagai penerus Santo Petrus, kepada siapa Yesus memberikan kunci Surga dan kuasa untuk ""mengikat dan melepaskan"" serta menyebutnya sebagai ""batu karang"" yang di atasnya Gereja kemudian dibangun. Paus saat ini adalah Paus Fransiskus, terpilih pada tanggal 13 Maret 2013, menggantikan Paus Benediktus XVI.[4]qa_0986961735e0c79ff572 tier 2",Paus,Pauscorrect,Fransiskuswrong_sentence,Paus Fransiskusboundary_too_long,Pauscorrect,Pauscorrect,(no answer)
4,"Siapa yang menemukan Tabel periodik?passageSejarah tabel periodik mencerminkan perkembangan pemahaman sifat-sifat kimia selama lebih dari satu abad. Peristiwa terpenting dalam sejarahnya terjadi pada tahun 1869, ketika tabel dipublikasikan oleh Dmitri Mendeleev,[1] yang mengembangkan tabel berdasarkan pengembangan terdahulu oleh ilmuwan seperti Antoine-Laurent de Lavoisier dan John Newlands, tetapi hanya Mendeleev yang mendapat kredit untuk pengembangannya.qa_0bf9cafef11db24ae688 tier 2",Dmitri Mendeleev,Dmitri Mendeleevcorrect,Dmitri Mendeleevcorrect,"Dmitri Mendeleev,[1] yang mengembangkan tabel berdasarkan pengembangan terdahulu oleh ilmuwan seperti Antoine-Laurent de Lavoisier dan John Newlandsboundary_too_long",Dmitri Mendeleevcorrect,n/a,n/a
5,"apakah nama ibukota Vietnam?passageIbukota Vietnam adalah Hanoi (dahulu berfungsi sebagai ibukota Vietnam Utara), sedangkan kota terbesar dan terpadat adalah Kota Ho Chi Minh (dahulu dikenal sebagai Saigon). provinsi (dalam Bahasa Vietnam di sebut tỉnh) dan 5 kotamadya yang di kontrol langsung oleh pemerintah pusat dan memiliki level yang sama dengan provinsi (thành phố trực thuộc trung ương). Ke-59 provinsi-provinsi tersebut kemudian dibagi-bagi menjadi kotamadya provinsi (thành phố trực thuộc tỉnh, daerah perkotaan (thị xã) dan pedesaan (huyện), dan kemudian dibagi lagi menjadi kota (thị trấn) atau komune (xã). Sedangkan, 5 kota madya yang dikontrol oleh pemerintah pusat di bagi menjadi distrik (quận) dan kabupaten, dan kemudian, dibagi lagi menjadi kelurahan (phường).qa_117ffbb8cd4ed821d9e9 tier 2",Hanoi,Kota Ho Chi Minhwrong_span_same_sentence,Hanoicorrect,"Hanoi (dahulu berfungsi sebagai ibukota Vietnam Utara), sedangkan kota terbesar dan terpadat adalah Kota Ho Chi Minhboundary_too_long",Hanoicorrect,n/a,n/a
6,"Apakah rumah adat Maluku?passageRumah Baileo adalah rumah adat Maluku dan Maluku Utara, Indonesia.[1] Rumah Baileo merupakan representasi kebudayaan Maluku dan memiliki fungsi yang sangat penting bagi kehidupan masyarakat.[1

## Error-type confusion
Rows and columns are the error categories from `mrc.analysis.compare.classify`
(`correct`, `false_no_answer`, `false_answer_on_unanswerable`, `wrong_sentence`, `wrong_span_same_sentence`,
`boundary_too_long`, `boundary_too_short`, `partial_overlap`). Cell [a, b] counts dev examples where model A made
error type a and model B made error type b on the same question (seed 13 unless stated).

- Large off-diagonal cells in a row show what A's errors turn into under B (for example, BiDAF `false_no_answer` that
  IndoBERT gets `correct` is what pretraining fixes).
- The diagonal minus `correct` counts shared errors, which point at the data (hard or mislabeled questions) rather than the model.
- `cm_ids(a, b, cat_a, cat_b)` returns the ids behind a cell; pass them to `side_by_side` or `show`.

In [9]:
CATS = ["correct", "false_no_answer", "false_answer_on_unanswerable", "wrong_sentence",
        "wrong_span_same_sentence", "boundary_too_long", "boundary_too_short", "partial_overlap"]
SHORT = {"correct": "correct", "false_no_answer": "false_noans", "false_answer_on_unanswerable": "false_ans_unans",
         "wrong_sentence": "wrong_sent", "wrong_span_same_sentence": "wrong_span_same_sent",
         "boundary_too_long": "too_long", "boundary_too_short": "too_short", "partial_overlap": "partial"}

def cat_of(r, m, seed="seed13"):
    if m in r["models"]:
        s = r["models"][m]["seeds"]
        return (s.get(seed) or next(iter(s.values())))["category"]
    p = r["partial_models"].get(m)
    return p["category"] if p else None

def confusion(a, b, seed_a="seed13", seed_b="seed13", ids=None):
    """Category crosstab: rows = model a, columns = model b, on the examples both cover."""
    pairs = [(cat_of(r, a, seed_a), cat_of(r, b, seed_b)) for r in R if ids is None or r["id"] in ids]
    pairs = [p for p in pairs if None not in p]
    t = pd.crosstab(pd.Series([p[0] for p in pairs], name=f"{a} ({seed_a})"),
                    pd.Series([p[1] for p in pairs], name=f"{b} ({seed_b})"))
    t = t.reindex(index=CATS, columns=CATS, fill_value=0)
    t.index, t.columns = [SHORT[c] for c in t.index], [SHORT[c] for c in t.columns]
    return t

def show_cm(t, title=""):
    shared = sum(t.iat[k, k] for k in range(1, len(t)))
    sty = (t.style.background_gradient(cmap="Oranges", axis=None)
           .set_caption(f"{title}  (n={t.values.sum()}, shared errors on diagonal={shared})"))
    display(sty)

def cm_ids(a, b, cat_a, cat_b, seed_a="seed13", seed_b="seed13"):
    long = {v: k for k, v in SHORT.items()}
    cat_a, cat_b = long.get(cat_a, cat_a), long.get(cat_b, cat_b)
    return [r["id"] for r in R if cat_of(r, a, seed_a) == cat_a and cat_of(r, b, seed_b) == cat_b]

### Outcome table: outcome type (rows) by model (columns)
Rows are grouped by gold: answerable questions end in exactly one of the answerable outcomes, unanswerable ones in
one of the two unanswerable outcomes, so each model's column sums to the group size. Values are the mean count over
seeds (std in the second table); the percent view divides by the group size.

In [10]:
OUTCOMES = [("answerable", "correct_ans"), ("answerable", "false_no_answer"), ("answerable", "wrong_sentence"),
            ("answerable", "wrong_span_same_sentence"), ("answerable", "boundary_too_long"),
            ("answerable", "boundary_too_short"), ("answerable", "partial_overlap"),
            ("unanswerable", "correct_no_ans"), ("unanswerable", "false_answer_on_unanswerable")]

def outcome(r, cat):
    if cat == "correct":
        return "correct_no_ans" if r["is_impossible"] else "correct_ans"
    return cat

def outcome_table(models=MAIN, ids=None):
    """Rows (gold, outcome), columns model; mean and std of counts over each model's seeds."""
    rs = [r for r in R if ids is None or r["id"] in ids]
    mean, std = {}, {}
    for m in models:
        per_seed = []
        for seed in rs[0]["models"][m]["seeds"]:
            c = collections.Counter(("unanswerable" if r["is_impossible"] else "answerable",
                                     outcome(r, r["models"][m]["seeds"][seed]["category"])) for r in rs)
            per_seed.append(pd.Series({k: c.get(k, 0) for k in OUTCOMES}))
        t = pd.concat(per_seed, axis=1)
        mean[m], std[m] = t.mean(axis=1), t.std(axis=1, ddof=0)
    idx = pd.MultiIndex.from_tuples(OUTCOMES, names=["gold", "outcome"])
    mean, std = pd.DataFrame(mean).reindex(idx), pd.DataFrame(std).reindex(idx)
    sizes = mean.groupby(level="gold").transform("sum")
    return mean, std, mean / sizes

mean_t, std_t, pct_t = outcome_table(["no_answer"] + MAIN)
assert (mean_t.groupby(level="gold").sum().round(6).nunique(axis=1) == 1).all(), "columns must sum to the group sizes"
display(mean_t.round(1).style.format("{:.1f}").background_gradient(cmap="Oranges", axis=None)
        .set_caption("mean count over seeds"))
display(pct_t.style.format("{:.1%}").background_gradient(cmap="Oranges", axis=None)
        .set_caption("share of the gold group (answerable n=%d, unanswerable n=%d)"
                     % tuple(mean_t.groupby(level="gold").sum().iloc[:, 0])))
display(std_t.style.format("{:.1f}").set_caption("std of counts over seeds (seed noise)"))

In [11]:
# Partial API LLM runs: same table on the ids they cover, untuned (n is tiny, read as anecdotes)
for m in API:
    rs = [r for r in R if m in r["partial_models"]]
    if not rs:
        continue
    c = collections.Counter(("unanswerable" if r["is_impossible"] else "answerable",
                             outcome(r, r["partial_models"][m]["category"])) for r in rs)
    base = {mm: collections.Counter(("unanswerable" if r["is_impossible"] else "answerable",
                                     outcome(r, cat_of(r, mm))) for r in rs) for mm in MAIN}
    t = pd.DataFrame({**{mm: [base[mm].get(k, 0) for k in OUTCOMES] for mm in MAIN},
                      m: [c.get(k, 0) for k in OUTCOMES]},
                     index=pd.MultiIndex.from_tuples(OUTCOMES, names=["gold", "outcome"]))
    display(t.style.background_gradient(cmap="Oranges", axis=None)
            .set_caption(f"{m} vs main models (seed 13) on the same {len(rs)} ids"))

In [12]:
# The main contrasts: what pretraining changes, what attention changes, classical vs neural
for a, b in [("bidaf", "plm_indobert_full"), ("bidaf_noattn", "bidaf"), ("classical_pipeline", "bidaf"),
             ("classical_pipeline", "plm_indobert_full")]:
    show_cm(confusion(a, b), f"{a} (rows) vs {b} (cols)")

,correct,false_noans,false_ans_unans,wrong_sent,wrong_span_same_sent,too_long,too_short,partial
correct,386,1,28,1,2,1,2,0
false_noans,103,88,0,15,8,14,11,9
false_ans_unans,15,0,7,0,0,0,0,0
wrong_sent,13,6,0,5,2,2,0,1
wrong_span_same_sent,7,3,0,0,0,2,0,0
too_long,30,5,0,1,3,7,0,1
too_short,4,1,0,2,0,0,6,0
partial,2,2,0,0,0,1,2,4


,correct,false_noans,false_ans_unans,wrong_sent,wrong_span_same_sent,too_long,too_short,partial
correct,376,22,7,5,2,13,3,0
false_noans,13,193,0,9,4,9,1,0
false_ans_unans,24,0,15,0,0,0,0,0
wrong_sent,2,8,0,12,1,3,1,1
wrong_span_same_sent,2,8,0,1,5,5,0,0
too_long,2,10,0,2,0,15,0,1
too_short,0,0,0,0,0,2,8,2
partial,2,7,0,0,0,0,0,7


,correct,false_noans,false_ans_unans,wrong_sent,wrong_span_same_sent,too_long,too_short,partial
correct,318,49,18,4,4,16,0,3
false_noans,33,128,0,13,6,18,3,3
false_ans_unans,59,0,4,0,0,0,0,0
wrong_sent,2,28,0,10,0,2,0,1
wrong_span_same_sent,3,21,0,1,1,7,0,2
too_long,0,3,0,0,1,1,1,0
too_short,6,12,0,1,0,2,8,1
partial,0,7,0,0,0,1,1,1


,correct,false_noans,false_ans_unans,wrong_sent,wrong_span_same_sent,too_long,too_short,partial
correct,369,9,22,2,1,8,0,1
false_noans,91,74,0,11,9,8,8,3
false_ans_unans,50,0,13,0,0,0,0,0
wrong_sent,16,11,0,8,1,2,0,5
wrong_span_same_sent,15,6,0,1,4,5,2,2
too_long,0,2,0,1,0,3,0,0
too_short,15,3,0,1,0,0,10,1
partial,4,1,0,0,0,1,1,3


In [13]:
# Seed stability: is a model's error type a property of the question or of the seed?
# A strong diagonal means errors are stable; mass off the diagonal means seed noise.
for m in MAIN:
    t = confusion(m, m, "seed13", "seed42")
    stable = np.trace(t.values) / t.values.sum()
    show_cm(t, f"{m}: seed13 (rows) vs seed42 (cols), same category {stable:.1%}")

,correct,false_noans,false_ans_unans,wrong_sent,wrong_span_same_sent,too_long,too_short,partial
correct,392,7,10,2,0,0,1,0
false_noans,7,191,0,3,0,2,0,1
false_ans_unans,6,0,57,0,0,0,0,0
wrong_sent,1,0,0,41,0,0,0,1
wrong_span_same_sent,0,0,0,4,29,1,1,0
too_long,0,0,0,0,0,6,0,0
too_short,1,1,0,1,1,0,25,1
partial,1,1,0,0,1,0,0,7


,correct,false_noans,false_ans_unans,wrong_sent,wrong_span_same_sent,too_long,too_short,partial
correct,380,3,26,10,1,2,5,1
false_noans,18,168,0,17,7,8,4,7
false_ans_unans,12,0,27,0,0,0,0,0
wrong_sent,1,0,0,19,2,4,1,1
wrong_span_same_sent,3,3,0,5,9,1,0,0
too_long,5,4,0,1,3,16,0,1
too_short,0,1,0,0,0,0,10,1
partial,2,0,0,3,0,1,1,9


,correct,false_noans,false_ans_unans,wrong_sent,wrong_span_same_sent,too_long,too_short,partial
correct,371,4,35,4,0,1,4,2
false_noans,24,172,0,23,8,11,3,7
false_ans_unans,4,0,18,0,0,0,0,0
wrong_sent,2,6,0,17,2,2,0,0
wrong_span_same_sent,4,1,0,2,5,0,0,0
too_long,11,5,0,2,2,21,2,4
too_short,1,2,0,0,0,0,10,0
partial,0,0,0,0,3,0,1,7


,correct,false_noans,false_ans_unans,wrong_sent,wrong_span_same_sent,too_long,too_short,partial
correct,528,8,13,2,3,4,2,0
false_noans,13,76,0,9,0,3,4,1
false_ans_unans,11,0,24,0,0,0,0,0
wrong_sent,2,1,0,17,2,1,1,0
wrong_span_same_sent,3,2,0,0,6,3,1,0
too_long,3,1,0,1,1,20,1,0
too_short,0,0,0,0,0,0,20,1
partial,2,0,0,0,1,1,0,11


In [14]:
# Shared-error rate for every pair: of the examples both get wrong, how often is it the same error type?
def same_err_rate(a, b):
    both = [(cat_of(r, a), cat_of(r, b)) for r in R]
    both = [p for p in both if p[0] != "correct" and p[1] != "correct"]
    return sum(x == y for x, y in both) / len(both) if both else float("nan"), len(both)

sm = pd.DataFrame({a: {b: same_err_rate(a, b)[0] for b in MAIN} for a in MAIN}).T
nb_ = pd.DataFrame({a: {b: same_err_rate(a, b)[1] for b in MAIN} for a in MAIN}).T
print("same error type | both wrong;  counts of both-wrong examples below")
display(sm.style.format("{:.0%}").background_gradient(cmap="Oranges", axis=None))
display(nb_)

same error type | both wrong;  counts of both-wrong examples below


,classical_pipeline,bidaf_noattn,bidaf,plm_indobert_full
classical_pipeline,100%,56%,53%,57%
bidaf_noattn,56%,100%,77%,61%
bidaf,53%,77%,100%,56%
plm_indobert_full,57%,61%,56%,100%


,classical_pipeline,bidaf_noattn,bidaf,plm_indobert_full
classical_pipeline,391,284,288,200
bidaf_noattn,284,375,330,210
bidaf,288,330,382,208
plm_indobert_full,200,210,208,243


In [15]:
# Answerability confusion (gold vs predicted), per model and seed 13: the over-abstention in one view
def answerability(m):
    g = pd.Series(["unanswerable" if r["is_impossible"] else "answerable" for r in R], name="gold")
    p = pd.Series(["no answer" if not (lambda s: (s.get("seed13") or next(iter(s.values())))["pred"])(r["models"][m]["seeds"])
                   else "answer" for r in R], name="predicts")
    return pd.crosstab(g, p).reindex(columns=["answer", "no answer"], fill_value=0)

display(pd.concat({m: answerability(m) for m in MAIN}, axis=1))

classical_pipeline           bidaf_noattn            bidaf  \
predicts                 answer no answer       answer no answer answer   
gold                                                                      
answerable                  228       204          203       229    184   
unanswerable                 63       308           39       332     22   

                       plm_indobert_full            
predicts     no answer            answer no answer  
gold                                                
answerable         248               326       106  
unanswerable       349                35       336

In [16]:
# Drill into one cell, e.g. BiDAF abstains but IndoBERT is right
ids = cm_ids("bidaf", "plm_indobert_full", "false_noans", "correct")
print(len(ids))
side_by_side(ids[:10])

103


#,question,gold,classical_pipeline,bidaf_noattn,bidaf,plm_indobert_full,llm_nemotron35_api,llm_gemini35flash_api
1,"Kapan Yan Xishan diangkat menjadi panglima perang Tiongkok ?passageYan Xishan (atau Yen Hsi-shan, IPA:[i̯ɛ́n ɕíʂan]; ) adalah seorang panglima perang yang mengabdi pada pemerintah Republik Tiongkok. Ia secara efektif memerintah Provinsi Shanxi sejak Revolusi Xinhai tahun 1911 hingga kemenangan Komunis dalam Perang Saudara Tiongkok tahun 1949. Selama menjadi pemimpin dari sebuah provinsi yang relatif kecil, miskin, dan terpencil, ia dapat bertahan menghadapi intrik Yuan Shikai, Era Pemimpin Perang, Era Nasionalis, invasi Jepang ke Tiongkok, dan perang saudara berkelanjutan. Kekuasaannya hanya terhenti setelah tentara Nasionalis, yang menjadi sekutunya, benar-benar kehilangan kontrol atas daratan Tiongkok, yang mengisolasi Shanxi dari semua sumber pasokan ekonomi dan militer. Ia dipandang oleh para penulis biografi Barat sebagai seorang tokoh transisi yang mendukung penggunaan teknologi Barat untuk melindungi tradisi Tionghoa, sementara pada saat yang sama perubahan kondisi politik, sosial, dan ekonomi sedang membuka jalan bagi perubahan radikal yang terjadi setelah pemerintahannya.[1]qa_00e198cdbe7a6db9fff7 tier 3",1911,pemerintah Republik Tiongkokwrong_sentence,(no answer),(no answer),1911correct,(no answer),(no answer)
2,"Apakah ibu kota Tanzania?passageSemula Tanzania bernama United Republic of Tanganyika and Zanibar, namun pada tanggal 26 April 1964 berubah menjadi United Republic of Tanzania, dengan ibukotanya yang sangat terkenal Dar-es-Salam.qa_032e963768b50c42794c tier 3",Dar-es-Salam,United Republic of Tanganyika and Zanibarwrong_span_same_sentence,(no answer),(no answer),Dar-es-Salamcorrect,Dar-es-Salamcorrect,Dar-es-Salamcorrect
3,"Apakah yang dimaksud dengan algoritme?passageDalam matematika dan ilmu komputer, algoritme adalah prosedur langkah-demi-langkah untuk penghitungan. Algoritme digunakan untuk penghitungan, pemrosesan data, dan penalaran otomatis.qa_0674f81bcf68276feae4 tier 3",prosedur langkah-demi-langkah untuk penghitungan,prosedur langkah-demi-langkah untuk penghitungancorrect,(no answer),(no answer),prosedur langkah-demi-langkah untuk penghitungancorrect,prosedur langkah-demi-langkah untuk penghitungancorrect,prosedur langkah-demi-langkah untuk penghitungancorrect
4,"Berapa luas Pulau Serangan?passageSerangan adalah sebuah kelurahan dan pulau yang berada di wilayah Kecamatan Denpasar Selatan, Kota Denpasar, Provinsi Bali, Indonesia. Pulau yang terletak 500 meter di sebelah selatan pusat kota ini memiliki memiliki panjang maksium 2,9 km dan lebar 1 km.qa_074634b2eae86659178f tier 3","panjang maksium 2,9 km dan lebar 1 km",(no answer),(no answer),(no answer),"panjang maksium 2,9 km dan lebar 1 kmcorrect",(no answer),(no answer)
5,"Sejak tanggal berapakah Jepang tidak berhak mencampuri urusan perkeretaapian di Indonesia?passagePada tanggal 28 September 1945, pembacaan pernyataan sikap oleh Ismangil dan sejumlah anggota AMKA lainnya menegaskan bahwa mulai hari itu kekuasaan perkeretaapian berada di tangan bangsa Indonesia sehingga Jepang sudah tidak berhak untuk mencampuri urusan perkeretaapian di Indonesia. Inilah yang melandasi ditetapkannya tanggal 28 September 1945 sebagai Hari Kereta Api serta dibentuknya Djawatan Kereta Api Repoeblik Indonesia (DKARI).qa_07b2bea085cd5a148a56 tier 3",28 September 1945,28 September 1945correct,(no answer),(no answer),28 September 1945correct,28 September 1945correct,28 September 1945correct
6,"apakah ibukota provinsi Kalimantan Selatan?passageKalimantan Selatan (disingkat Kalsel) adalah salah satu provinsi di Indonesia yang terletak di pulau Kalimantan. Ibu kotanya adalah Banjarmasin. Provinsi Kalimantan Selatan memiliki luas 37.530,52km²[1] dengan populasi hampir 3,7 juta jiwa.qa_0a266524498967311c7b tier 3",Banjarmasin,pulau Kalimantanwrong_sentence,(no answer),(no answer),Banjarmasincorrect,Banjarmasincorrect,n/a
7,"Kapan Minecraft pertama kali diciptak

In [17]:
# Optional: the same confusion restricted to a slice, e.g. long answers
long_ids = set(df.index[df.ans_len_bucket == ">6"])
show_cm(confusion("bidaf", "plm_indobert_full", ids=long_ids), "long answers: bidaf vs plm_indobert_full")

,correct,false_noans,false_ans_unans,wrong_sent,wrong_span_same_sent,too_long,too_short,partial
correct,10,0,0,0,0,0,2,0
false_noans,15,23,0,1,2,2,8,2
false_ans_unans,0,0,0,0,0,0,0,0
wrong_sent,1,1,0,0,0,1,0,0
wrong_span_same_sent,0,0,0,0,0,0,0,0
too_long,4,0,0,0,0,0,0,0
too_short,4,1,0,1,0,0,5,0
partial,1,1,0,0,0,0,2,4


## Queries
Edit and rerun. `df` columns: `<model>.f1` (mean over seeds), `<model>.cat` and `<model>.pred` (seed 13), plus slice tags.

In [18]:
# 1. Hard set: every main model wrong (mean F1 < 0.5)
hard = df[(df[[f"{m}.f1" for m in MAIN]] < 0.5).all(axis=1)]
print(len(hard), "hard examples;", hard.is_impossible.mean().round(3), "unanswerable share")
show(hard.head(5))

133 hard examples; 0.015 unanswerable share


classical_pipeline seed13,wrong_span_same_sentence F1 0.00,"Semula Tanzania bernama United Republic of Tanganyika and Zanibar, namun pada tanggal 26 April 1964 berubah menjadi United Republic of ..."
bidaf_noattn seed13,false_no_answer F1 0.00,no answer
bidaf seed13,false_no_answer F1 0.00,no answer
plm_indobert_full seed13,correct F1 1.00,"...i United Republic of Tanzania, dengan ibukotanya yang sangat terkenal Dar-es-Salam."
llm_nemotron35_api exact,correct F1 1.00,"...i United Republic of Tanzania, dengan ibukotanya yang sangat terkenal Dar-es-Salam.raw: Dar-es-Salam"
llm_gemini35flash_api exact,correct F1 1.00,"...i United Republic of Tanzania, dengan ibukotanya yang sangat terkenal Dar-es-Salam.raw: Dar-es-Salam"
classical_pipeline seed13,false_no_answer F1 0.00,no answer
bidaf_noattn seed13,false_no_answer F1 0.00,no answer
bidaf seed13,wrong_sentence F1 0.00,NamaMasa jabatan1. Hammer DeRoburt (1922-1992)31 Januari 1968 - 22 Desember 19762. Bern...
plm_indobert_full seed13,false_no_answer F1 0.00,no answer
llm_nemotron35_api exact,boundary_too_long F1 0.80,NamaMasa jabatan1. Hammer DeRoburt (1922-1992)31 Januari 1968 - 22 Desember 19762. Bernard Dowiyogo (1946-2003)22 De...raw: Hammer DeRoburt (1922-1992)


In [19]:
# 2. Where pretraining helps: IndoBERT right, BiDAF wrong
only_plm = df[(df["plm_indobert_full.f1"] >= 0.5) & (df["bidaf.f1"] < 0.5)]
print(len(only_plm))
show(only_plm.sample(min(5, len(only_plm)), random_state=0))

181


classical_pipeline seed13,false_no_answer F1 0.00,no answer
bidaf_noattn seed13,false_no_answer F1 0.00,no answer
bidaf seed13,boundary_too_long F1 0.12,"...patan diturunkan menjadi 21% pada bulan Januari 2008.[125] Pemerintah Republik Estonia merampungkan desain uang logam euro Estonia pada akhir tahun 2004, dan mengadopsi euro sebagai mata uang negara ini sejak tanggal 1 Januari 2011, lebih lamb..."
plm_indobert_full seed13,correct F1 1.00,"... desain uang logam euro Estonia pada akhir tahun 2004, dan mengadopsi euro sebagai mata uang negara ini sejak tanggal 1 Januari 2011, lebih lamb..."
classical_pipeline seed13,wrong_sentence F1 0.00,...gkat Kalsel) adalah salah satu provinsi di Indonesia yang terletak di pulau Kalimantan. Ibu kotanya adalah Banjarmasin. Provinsi Kalimantan Selatan memiliki...
bidaf_noattn seed13,false_no_answer F1 0.00,no answer
bidaf seed13,false_no_answer F1 0.00,no answer
plm_indobert_full seed13,correct F1 1.00,"...si di Indonesia yang terletak di pulau Kalimantan. Ibu kotanya adalah Banjarmasin. Provinsi Kalimantan Selatan memiliki luas 37.530,52km²[1] dengan pop..."
llm_nemotron35_api exact,correct F1 1.00,"...si di Indonesia yang terletak di pulau Kalimantan. Ibu kotanya adalah Banjarmasin. Provinsi Kalimantan Selatan memiliki luas 37.530,52km²[1] dengan pop...raw: Banjarmasin"
classical_pipeline seed13,correct F1 1.00,Adam Smith sering disebut sebagai yang pertama mengembangkan ilmu ekonomi pada a...
bidaf_noattn seed13,wrong_span_same_sentence F1 0.00,"...angkan ilmu ekonomi pada abad 18 sebagai satu cabang tersendiri dalam ilmu pengetahuan. Melalui karya besarnya , Smith mencoba mencari tahu sejarah perkemba..."


In [20]:
# 3. Where feature engineering wins: classical right, IndoBERT wrong
only_cls = df[(df["classical_pipeline.f1"] >= 0.5) & (df["plm_indobert_full.f1"] < 0.5)]
print(len(only_cls))
show(only_cls.head(5))

42


classical_pipeline seed13,correct F1 1.00,no answer
bidaf_noattn seed13,correct F1 1.00,no answer
bidaf seed13,correct F1 1.00,no answer
plm_indobert_full seed13,false_answer_on_unanswerable F1 0.00,...mbuhan yang kehadirannya tidak diinginkan pada lahan pertanian karena menurunkan hasil yang bisa dicapai oleh tanaman produksi.
classical_pipeline seed13,correct F1 1.00,no answer
bidaf_noattn seed13,correct F1 1.00,no answer
bidaf seed13,correct F1 1.00,no answer
plm_indobert_full seed13,false_answer_on_unanswerable F1 0.00,"Pada tanggal 18 Februari 1965, Gambia diberi kemerdekaan oleh Britania Raya lalu bergabung dalam Pe..."
classical_pipeline seed13,correct F1 1.00,"Rumah Baileo adalah rumah adat Maluku dan Maluku Utara, Indonesia.[1] Rumah Baileo..."
bidaf_noattn seed13,partial_overlap F1 0.22,"Rumah Baileo adalah rumah adat Maluku dan Maluku Utara, Indonesia.[1] Rumah Baileo merupakan representasi kebudayaan Maluku dan memilik..."
bidaf seed13,partial_overlap F1 0.22,"Rumah Baileo adalah rumah adat Maluku dan Maluku Utara, Indonesia.[1] Rumah Baileo merupakan representasi kebudayaan Maluku dan memilik..."


In [21]:
# 4. One error category for one model, e.g. IndoBERT false no-answers (all seeds shown)
fna = df[df["plm_indobert_full.cat"] == "false_no_answer"]
print(len(fna))
show(fna.head(4), seeds=None)

106


classical_pipeline seed13,false_no_answer F1 0.00,no answer
classical_pipeline seed42,false_no_answer F1 0.00,no answer
classical_pipeline seed87,false_no_answer F1 0.00,no answer
bidaf_noattn seed13,boundary_too_short F1 0.50,"... Hendrik Prai, berencana mendeklarasikan kemerdekaan Papua pada tahun 1971. Tanggal 1 Juli 1971, Roemkorem dan Prai mendeklarasikan Republik Pap..."
bidaf_noattn seed42,boundary_too_short F1 0.50,"... Hendrik Prai, berencana mendeklarasikan kemerdekaan Papua pada tahun 1971. Tanggal 1 Juli 1971, Roemkorem dan Prai mendeklarasikan Republik Pap..."
bidaf_noattn seed87,boundary_too_long F1 0.75,"... Hendrik Prai, berencana mendeklarasikan kemerdekaan Papua pada tahun 1971. Tanggal 1 Juli 1971, Roemkorem dan Prai mendeklarasikan Republik Papua Barat dan segera m..."
bidaf seed13,boundary_too_long F1 0.75,"... Hendrik Prai, berencana mendeklarasikan kemerdekaan Papua pada tahun 1971. Tanggal 1 Juli 1971, Roemkorem dan Prai mendeklarasikan Republik Papua Barat dan segera m..."
bidaf seed42,boundary_too_long F1 0.75,"... Hendrik Prai, berencana mendeklarasikan kemerdekaan Papua pada tahun 1971. Tanggal 1 Juli 1971, Roemkorem dan Prai mendeklarasikan Republik Papua Barat dan segera m..."
bidaf seed87,boundary_too_short F1 0.50,"... Hendrik Prai, berencana mendeklarasikan kemerdekaan Papua pada tahun 1971. Tanggal 1 Juli 1971, Roemkorem dan Prai mendeklarasikan Republik Pap..."
plm_indobert_full seed13,false_no_answer F1 0.00,no answer
plm_indobert_full seed42,false_no_answer F1 0.00,no answer


In [22]:
# 5. Slice drill-down, e.g. long answers (>6 words) or questions ending with " ?"
show(df[(df.ans_len_bucket == ">6") & (df["plm_indobert_full.f1"] < 0.5)].head(4))

classical_pipeline seed13,false_no_answer F1 0.00,no answer
bidaf_noattn seed13,boundary_too_short F1 0.18,"...da tahun 2003, World Conservation Union mencatat 147 spesies mamalia, 114 burung, 91 ikan dan 2 invertebrata termasuk dalam hewan-hewan yang te..."
bidaf seed13,boundary_too_short F1 0.18,"...da tahun 2003, World Conservation Union mencatat 147 spesies mamalia, 114 burung, 91 ikan dan 2 invertebrata termasuk dalam hewan-hewan yang te..."
plm_indobert_full seed13,boundary_too_short F1 0.18,"...m perjalanan.[12]. Pada tahun 2003, World Conservation Union mencatat 147 spesies mamalia, 114 burung, 91 ikan dan 2 invertebrata termasuk dala..."
classical_pipeline seed13,wrong_sentence F1 0.00,"..., seperti 2,4871773339… atau 3,25678. Bilangan real meliputi bilangan rasional, seperti 42 dan −23/129, dan bilangan irasional, seperti π dan ..."
bidaf_noattn seed13,false_no_answer F1 0.00,no answer
bidaf seed13,false_no_answer F1 0.00,no answer
plm_indobert_full seed13,wrong_span_same_sentence F1 0.00,"...enyatakan bilangan yang bisa dituliskan dalam bentuk desimal, seperti 2,4871773339… atau 3,25678. Bilangan real meliputi bilangan rasional, seperti 42 dan −23/129, da..."
classical_pipeline seed13,boundary_too_short F1 0.36,"...esuai dengan ibukotanya, Babilon, adalah negara kuno yang terletak di selatan Mesopotamia (sekarang Irak), di wilayah Sumeria dan Akkadia. Babel pertama disebu..."
bidaf_noattn seed13,correct F1 1.00,"...esuai dengan ibukotanya, Babilon, adalah negara kuno yang terletak di selatan Mesopotamia (sekarang Irak), di wilayah Sumeria dan Akkadia. Babel pertama disebut dalam sebuah tablet dari masa pemerintahan Sar..."
bidaf seed13,correct F1 1.00,"...esuai dengan ibukotanya, Babilon, adalah negara kuno yang terletak di selatan Mesopotamia (sekarang Irak), di wilayah Sumeria dan Akkadia. Babel pertama disebut dalam sebuah tablet dari masa pemerintahan Sar..."


In [23]:
# 6. The partial API LLM runs next to the others
api = [i for i in df.index if BY_ID[i]["partial_models"]]
show(api[:6])

classical_pipeline seed13,correct F1 1.00,"... Perjanjian Roma tahun 1957.[1] Setelah Uni Eropa (UE) dibentuk tahun 1993, MEE disatukan dan berganti nama menjadi Masyarakat Eropa (EC). Pada ..."
bidaf_noattn seed13,wrong_sentence F1 0.00,"...ara anggotanya. Organisasi ini dibentuk melalui Perjanjian Roma tahun 1957.[1] Setelah Uni Eropa (UE) dibentuk tahun 1993, MEE disatukan dan ber..."
bidaf seed13,wrong_sentence F1 0.00,"...ara anggotanya. Organisasi ini dibentuk melalui Perjanjian Roma tahun 1957.[1] Setelah Uni Eropa (UE) dibentuk tahun 1993, MEE disatukan dan ber..."
plm_indobert_full seed13,correct F1 1.00,"... Perjanjian Roma tahun 1957.[1] Setelah Uni Eropa (UE) dibentuk tahun 1993, MEE disatukan dan berganti nama menjadi Masyarakat Eropa (EC). Pada ..."
llm_nemotron35_api exact,boundary_too_long F1 0.67,"...elalui Perjanjian Roma tahun 1957.[1] Setelah Uni Eropa (UE) dibentuk tahun 1993, MEE disatukan dan berganti nama menjadi Masyarakat Eropa (EC). Pada ...raw: tahun 1993"
llm_gemini35flash_api noans,false_no_answer F1 0.00,no answerraw: TIDAK ADA JAWABAN
classical_pipeline seed13,wrong_sentence F1 0.00,...PA:[i̯ɛ́n ɕíʂan]; ) adalah seorang panglima perang yang mengabdi pada pemerintah Republik Tiongkok. Ia secara efektif memerintah Provinsi Shanxi sejak Revolusi Xinhai t...
bidaf_noattn seed13,false_no_answer F1 0.00,no answer
bidaf seed13,false_no_answer F1 0.00,no answer
plm_indobert_full seed13,correct F1 1.00,...secara efektif memerintah Provinsi Shanxi sejak Revolusi Xinhai tahun 1911 hingga kemenangan Komunis dalam Perang Saudara Tiongkok tahun 1949. S...
llm_nemotron35_api noans,false_no_answer F1 0.00,no answerraw: TIDAK ADA JAWABAN


In [24]:
# 7. Export any selection for manual labeling
# hard.assign(human_label="", notes="").to_csv(ROOT / "eda/outputs/tables/hard_set_for_labeling.csv", encoding="utf-8")